In [1]:
from simbanator.io.simba import Simulation
from simbanator.analysis.particles import extract_particles
from astropy.io import fits
import numpy as np
# Load your simulation (use the name you configured)
sim = Simulation("cis100")  # Replace XX with your snapshot number

In [5]:
from pathlib import Path
import textwrap
import subprocess

targetfile = '/mnt/home/glorenzon/output/quenching_times/legacy_zsp_sampled_snapshots_with_ids.fits'
with fits.open(targetfile) as hdul:
    data = hdul[1].data
    id_draws = np.asarray(data['ID_AT_SELECTED_SNAP'], dtype=np.int64)
    snaps = np.asarray(data['SNAP_SELECTED'], dtype=np.int64)
    print(snaps)

# unique (snapshot, galaxy_id) pairs -> one array task per pair
jobs = np.unique(np.column_stack([snaps, id_draws]), axis=0)
if jobs.size == 0:
    raise RuntimeError('No jobs found in FITS table.')

job_dir = Path.cwd() / 'output' / 'slurm' / 'filter_particles'
log_dir = job_dir / 'logs'
job_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

manifest_path = job_dir / 'jobs.tsv'
np.savetxt(
    manifest_path,
    jobs,
    fmt='%d\t%d',
    header='snap\tgalaxy_id',
    comments=''
 )

worker_path = job_dir / 'run_filter_particles_worker.py'
worker_path.write_text(
    textwrap.dedent(
        '''
        import argparse
        from pathlib import Path
        import numpy as np
        from simbanator.io.simba import Simulation
        from simbanator.analysis.particles import extract_particles

        def main():
            parser = argparse.ArgumentParser()
            parser.add_argument('--manifest', required=True)
            parser.add_argument('--task-id', type=int, required=True)
            parser.add_argument('--simulation', default='cis100')
            args = parser.parse_args()

            rows = np.loadtxt(args.manifest, dtype=np.int64, skiprows=1)
            if rows.ndim == 1:
                rows = rows[np.newaxis, :]

            if args.task_id < 0 or args.task_id >= len(rows):
                raise IndexError(
                    f'task_id={args.task_id} is out of range for {len(rows)} jobs'
                )

            snap, galaxy_id = rows[args.task_id]
            sim = Simulation(args.simulation)
            obj = sim.load_catalog(int(snap))
            snap_path = sim.get_snapshot_file(int(snap))

            snap_dir = Path.cwd() / 'output' / args.simulation / 'filtered' / f'snap_{int(snap):03d}'
            snap_dir.mkdir(parents=True, exist_ok=True)

            output_file = snap_dir / f'snap_m100n1024_{int(snap):03d}_snap{int(snap)}_gal{int(galaxy_id)}.h5'

            extract_particles(
                obj,
                snap_path,
                snap=int(snap),
                galaxy_id=int(galaxy_id),
                output=str(output_file),
            )
            print(f'Completed snap={int(snap)} galaxy_id={int(galaxy_id)} -> {output_file}')

        if __name__ == '__main__':
            main()
        '''
    ).strip() + '\n'
 )

slurm_path = job_dir / 'submit_filter_particles_array.sh'
slurm_path.write_text(
    textwrap.dedent(
        f'''
        #!/bin/bash
        #SBATCH --job-name=flt_parts
        #SBATCH --output={log_dir}/%x_%A_%a.out
        #SBATCH --error={log_dir}/%x_%A_%a.err
        #SBATCH --time=04:00:00
        #SBATCH --cpus-per-task=1
        #SBATCH --mem=8G
        #SBATCH --array=0-{len(jobs)-1}

        set -euo pipefail
        cd {Path.cwd()}
        source $HOME/miniconda3/etc/profile.d/conda.sh
        conda activate pd39
        python {worker_path} --manifest {manifest_path} --task-id ${{SLURM_ARRAY_TASK_ID}}
        '''
    ).strip() + '\n'
 )
slurm_path.chmod(0o750)

submit_jobs = False  # flip to True to submit now
if submit_jobs:
    res = subprocess.run(['sbatch', str(slurm_path)], check=True, text=True, capture_output=True)
    print(res.stdout.strip())
else:
    print(f'Prepared {len(jobs)} array tasks.')
    print(f'Manifest: {manifest_path}')
    print(f'SLURM script: {slurm_path}')
    print(f'To submit: sbatch {slurm_path}')
    print('Outputs will be written under output/<simulation>/filtered/snap_XXX/')

[117 120 105 115 109 114 110 112 109 117 105 109 116 116 111 120 111 111
 107  99 123 114 110 123 116 100 112 105 115  97  99 106 114 117 115 122
 105 110 117 105 110 115 112 110 105 109 118 115 112 107 114 112 118 117
 104 102 108 119 117 103 106 114 120 111 114 117 107 115 107 117 116 110
 105 105 119 108 106 116 113 114 107 114 108 104 109 112 108 116 117 117
 114 109 108 115 117 111 117 117  94 114 109 111 109 106 120 105 116 111
 108 114 117 115 116 117 103 108 108 114 113 110 117 117 105 118 117 110
 118 115 115 110 110 107 107 109 110 106 111 115 106 108 110 119 117 118
  96 115 117 114 114 114  99 117 106 113 109 102 106 107 117 114 117 100
 120 107 120 107 108 106 117 108 117 113 105 116 107 118 114 114 106 109
 116 118 103 107 114 106 116 115 116 115 119 129 110 107 109 116 108 112
 117 109 115 106 114 119 106 110 108 112 105 114 115 114 116 107 116 108
 118 114 117  99 106  97 119 110 109 117 108 106 108 110 108 111 117 115
 117 116 120 109 115 117 108  99 112 117 114 117 11